------------------------------

Import the important libraries required for data processing\
 model building, training, evaluation, and cross-validation.

------------------------------------------------

In [ ]:
import joblib
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
from Mymodel import load_data, compile_model, train_and_save_model, evaluate_model
from mymodel_ import load_kfold_data, run_kfold_multiple_architectures

-------------------------

Load the pre-saved train, validation, and test datasets and display their shapes to verify correct loading.


-------------------------

In [ ]:
split_dir = r"C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits"  
X_train, y_train, X_valid, y_valid, X_test, y_test = load_data(split_dir)
print("Train set:", X_train.shape, y_train.shape)
print("Valid set:", X_valid.shape, y_valid.shape)
print("Test set:", X_test.shape, y_test.shape)


Train set: (81454, 600, 1) (81454,)
Valid set: (11636, 600, 1) (11636,)
Test set: (23273, 600, 1) (23273,)


In [ ]:
data_path = r"C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\processed_data_kfold"
X_array, y_array, le = load_kfold_data(data_path)

--------------------

Load the K-Fold dataset, reshape/convert X into Conv1D-ready format, and print shapes plus the number of classes to verify everything is prepared correctly.


--------------------------------

In [5]:
from mymodel_ import load_kfold_data, run_kfold_multiple_architectures
import numpy as np

data_path = r"C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\processed_data_kfold"
X_array, y_array, le = load_kfold_data(data_path)

# Conv1D wants (N, T, 1)
if X_array.ndim == 2:
    X_array = X_array.reshape((X_array.shape[0], X_array.shape[1], 1))
X_array = X_array.astype("float32", copy=False)

print("✅ X_array ready for Conv1D:", X_array.shape)
print("✅ y_array shape:", y_array.shape)
print("✅ num classes:", len(le.classes_))
 

✅ X_array ready for Conv1D: (132306, 600, 1)
✅ y_array shape: (132306,)
✅ num classes: 18


In [ ]:
X_array, y_array, le = load_kfold_data(data_path)
if X_array.ndim == 2:
    X_array = X_array[..., None]   # (N, 600) -> (N, 600, 1)
print("X_array shape:", X_array.shape)
print("y_array shape:", y_array.shape)


X_array shape: (132306, 600, 1)
y_array shape: (132306,)


--------------------------------------------------

Load the K-Fold dataset, reshape inputs for Conv1D, define multiple CNN architectures, and run 5-fold cross-validation to train and compare them while saving the results.


----------------------------------------------------------

In [ ]:
data_path = r"C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\processed_data_kfold"
X_array, y_array, le = load_kfold_data(data_path)

# ---- FIX 1: Make X compatible with Conv1D ----
# Conv1D expects (N, timesteps, channels). the  data is likely (N, 600).
if X_array.ndim == 2:
    X_array = X_array[..., None]  # (N, 600) -> (N, 600, 1)

print("X_array shape:", X_array.shape)  # should be (N, 600, 1)
print("y_array shape:", y_array.shape)

def build_cnn_small_1(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_small_2(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.Conv1D(128, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_medium_1(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(128, 5, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_medium_2(input_shape, num_classes):
    # (No reshape needed here anymore because we already reshaped X_array)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(128, 5, activation='relu'),
        tf.keras.layers.Conv1D(128, 3, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_big_2(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(256, 7, activation='relu'),
        tf.keras.layers.Conv1D(256, 5, activation='relu'),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(512, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


architectures = {
    "cnn_small_1": build_cnn_small_1,
    "cnn_small_2": build_cnn_small_2,
    "cnn_medium_1": build_cnn_medium_1,
    "cnn_medium_2": build_cnn_medium_2,
    "cnn_big_2": build_cnn_big_2
}

run_kfold_multiple_architectures(
    X_array=X_array,
    y_array=y_array,
    label_encoder=le,
    architectures_dict=architectures,
    n_splits=5,
    epochs=300,
    batch_size=2048,
    output_root="kfold_group_classification_all_architectures_aromatics"
)


---------------------------------------

Load the K-Fold dataset, define a single CNN architecture, and run 5-fold cross-validation on it to evaluate its performance and save the results.


-----------------------------------

In [ ]:
data_path = r"C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\processed_data_kfold"
X_array, y_array, le = load_kfold_data(data_path)
def cnn_full_architecture(input_shape, num_classes):
    from tensorflow.keras import models, layers, optimizers

    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv1D(128, kernel_size=5, activation='relu'),
        layers.Conv1D(128, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2, strides=2),

        layers.Conv1D(256, kernel_size=3, activation='relu'),
        layers.Conv1D(256, kernel_size=3, activation='relu'),

        layers.GlobalMaxPooling1D(),

        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Register the architecture
architectures = {
    "cnn_full": cnn_full_architecture
}

# Run the K-Fold experiment
run_kfold_multiple_architectures(
    X_array=X_array,
    y_array=y_array,
    label_encoder=le,
    architectures_dict=architectures,
    n_splits=5,
    epochs=300,
    batch_size=128,
    output_root="kfold_group_classification_Aromatics"
)



================== Training architecture: cnn_full ==================

🔁 Fold 1/5 — cnn_full
Epoch 1/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4526 - loss: 2.3223

655/655 ━━━━━━━━━━━━━━━━━━━━ 693s 1s/step - accuracy: 0.4528 - loss: 2.3208 - val_accuracy: 0.6384 - val_loss: 0.9197
Epoch 2/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6616 - loss: 0.8873

655/655 ━━━━━━━━━━━━━━━━━━━━ 693s 1s/step - accuracy: 0.6616 - loss: 0.8873 - val_accuracy: 0.6894 - val_loss: 0.8098
Epoch 3/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 0s 972ms/step - accuracy: 0.7095 - loss: 0.7669

655/655 ━━━━━━━━━━━━━━━━━━━━ 652s 996ms/step - accuracy: 0.7095 - loss: 0.7669 - val_accuracy: 0.6970 - val_loss: 0.8071
Epoch 4/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 0s 957ms/step - accuracy: 0.7366 - loss: 0.6990

655/655 ━━━━━━━━━━━━━━━━━━━━ 642s 980ms/step - accuracy: 0.7366 - loss: 0.6990 - val_accuracy: 0.7246 - val_loss: 0.6993
Epoch 5/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 640s 976ms/step - accuracy: 0.7580 - loss: 0.6442 - val_accuracy: 0.7164 - val_loss: 0.7394
Epoch 6/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 0s 951ms/step - accuracy: 0.7711 - loss: 0.6124

...

655/655 ━━━━━━━━━━━━━━━━━━━━ 765s 1s/step - accuracy: 0.9120 - loss: 0.2346 - val_accuracy: 0.8201 - val_loss: 0.5558
Epoch 25/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9142 - loss: 0.2291

655/655 ━━━━━━━━━━━━━━━━━━━━ 771s 1s/step - accuracy: 0.9142 - loss: 0.2291 - val_accuracy: 0.8203 - val_loss: 0.5933
Epoch 26/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 767s 1s/step - accuracy: 0.9184 - loss: 0.2203 - val_accuracy: 0.8168 - val_loss: 0.6242
Epoch 27/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 762s 1s/step - accuracy: 0.9234 - loss: 0.2095 - val_accuracy: 0.8137 - val_loss: 0.6564
Epoch 28/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 759s 1s/step - accuracy: 0.9249 - loss: 0.2052 - val_accuracy: 0.8131 - val_loss: 0.6730
Epoch 29/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9305 - loss: 0.1895

655/655 ━━━━━━━━━━━━━━━━━━━━ 761s 1s/step - accuracy: 0.9305 - loss: 0.1895 - val_accuracy: 0.8228 - val_loss: 0.6246
Epoch 30/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 757s 1s/step - accuracy: 0.9306 - loss: 0.1845 - val_accuracy: 0.8194 - val_loss: 0.6577
Epoch 31/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 757s 1s/step - accuracy: 0.9327 - loss: 0.1818 - val_accuracy: 0.8200 - val_loss: 0.5795
Epoch 32/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 754s 1s/step - accuracy: 0.9387 - loss: 0.1688 - val_accuracy: 0.8191 - val_loss: 0.6872
Epoch 33/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 769s 1s/step - accuracy: 0.9401 - loss: 0.1668 - val_accuracy: 0.8193 - val_loss: 0.6399
Epoch 34/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9419 - loss: 0.1597

655/655 ━━━━━━━━━━━━━━━━━━━━ 765s 1s/step - accuracy: 0.9419 - loss: 0.1597 - val_accuracy: 0.8254 - val_loss: 0.6837
Epoch 35/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 761s 1s/step - accuracy: 0.9436 - loss: 0.1547 - val_accuracy: 0.8254 - val_loss: 0.6335
Epoch 36/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 756s 1s/step - accuracy: 0.9483 - loss: 0.1397 - val_accuracy: 0.8175 - val_loss: 0.6947
Epoch 37/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 752s 1s/step - accuracy: 0.9455 - loss: 0.1511 - val_accuracy: 0.8240 - val_loss: 0.7534
Epoch 38/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 756s 1s/step - accuracy: 0.9489 - loss: 0.1377 - val_accuracy: 0.8253 - val_loss: 0.7099
Epoch 39/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 783s 1s/step - accuracy: 0.9507 - loss: 0.1362 - val_accuracy: 0.8207 - val_loss: 0.7156
Epoch 40/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9483 - loss: 0.1397

655/655 ━━━━━━━━━━━━━━━━━━━━ 766s 1s/step - accuracy: 0.9483 - loss: 0.1397 - val_accuracy: 0.8313 - val_loss: 0.7677
Epoch 41/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 761s 1s/step - accuracy: 0.9524 - loss: 0.1301 - val_accuracy: 0.8196 - val_loss: 0.7325
Epoch 42/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 752s 1s/step - accuracy: 0.9532 - loss: 0.1276 - val_accuracy: 0.8244 - val_loss: 0.7927
Epoch 43/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 757s 1s/step - accuracy: 0.9563 - loss: 0.1220 - val_accuracy: 0.8165 - val_loss: 0.8086
Epoch 44/300
655/655 ━━━━━━━━━━━━━━━━━━━━ 758s 1s/step - accuracy: 0.9566 - loss: 0.1191 - val_accuracy: 0.8109 - val_loss: 0.8943
Epoch 45/300
507/655 ━━━━━━━━━━━━━━━━━━━━ 2:47 1s/step - accuracy: 0.9567 - loss: 0.1192

--------------------------------------------------------------

In [ ]:

data_path = r"C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\processed_data_kfold"
X_array, y_array, le = load_kfold_data(data_path)
def cnn_full_architecture(input_shape, num_classes):
    from tensorflow.keras import models, layers, optimizers

    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv1D(128, kernel_size=5, activation='relu'),
        layers.Conv1D(128, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2, strides=2),

        layers.Conv1D(256, kernel_size=3, activation='relu'),
        layers.Conv1D(256, kernel_size=3, activation='relu'),

        layers.GlobalMaxPooling1D(),

        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Register the architecture
architectures = {
    "cnn_full": cnn_full_architecture
}

# Run the K-Fold experiment
run_kfold_multiple_architectures(
    X_array=X_array,
    y_array=y_array,
    label_encoder=le,
    architectures_dict=architectures,
    n_splits=3,
    epochs=30,
    batch_size=128,
    output_root="kfold_group_classification_Aromatics_second_try"
)



================== Training architecture: cnn_full ==================

🔁 Fold 1/3 — cnn_full
Epoch 1/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 999ms/step - accuracy: 0.4395 - loss: 2.6096 

546/546 ━━━━━━━━━━━━━━━━━━━━ 561s 1s/step - accuracy: 0.4396 - loss: 2.6075 - val_accuracy: 0.6445 - val_loss: 0.9516
Epoch 2/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6358 - loss: 0.9460

546/546 ━━━━━━━━━━━━━━━━━━━━ 570s 1s/step - accuracy: 0.6359 - loss: 0.9459 - val_accuracy: 0.6794 - val_loss: 0.8283
Epoch 3/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6999 - loss: 0.7863

546/546 ━━━━━━━━━━━━━━━━━━━━ 566s 1s/step - accuracy: 0.6999 - loss: 0.7863 - val_accuracy: 0.6845 - val_loss: 0.8322
Epoch 4/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7291 - loss: 0.7164

546/546 ━━━━━━━━━━━━━━━━━━━━ 589s 1s/step - accuracy: 0.7291 - loss: 0.7164 - val_accuracy: 0.7128 - val_loss: 0.7428
Epoch 5/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 608s 1s/step - accuracy: 0.7496 - loss: 0.6675 - val_accuracy: 0.7096 - val_loss: 0.7600
Epoch 6/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7623 - loss: 0.6258

...

546/546 ━━━━━━━━━━━━━━━━━━━━ 553s 1s/step - accuracy: 0.8741 - loss: 0.3357 - val_accuracy: 0.7873 - val_loss: 0.6436
Epoch 21/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 993ms/step - accuracy: 0.8825 - loss: 0.3133

546/546 ━━━━━━━━━━━━━━━━━━━━ 555s 1s/step - accuracy: 0.8825 - loss: 0.3133 - val_accuracy: 0.7895 - val_loss: 0.5863
Epoch 22/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 557s 1s/step - accuracy: 0.8867 - loss: 0.3026 - val_accuracy: 0.7886 - val_loss: 0.6072
Epoch 23/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 554s 1s/step - accuracy: 0.8927 - loss: 0.2900 - val_accuracy: 0.7877 - val_loss: 0.6601
Epoch 24/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 555s 1s/step - accuracy: 0.8982 - loss: 0.2705 - val_accuracy: 0.7855 - val_loss: 0.6514
Epoch 25/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 995ms/step - accuracy: 0.9011 - loss: 0.2623

546/546 ━━━━━━━━━━━━━━━━━━━━ 558s 1s/step - accuracy: 0.9011 - loss: 0.2623 - val_accuracy: 0.7974 - val_loss: 0.6285
Epoch 26/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 558s 1s/step - accuracy: 0.9077 - loss: 0.2504 - val_accuracy: 0.7854 - val_loss: 0.6851
Epoch 27/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 994ms/step - accuracy: 0.9098 - loss: 0.2402

546/546 ━━━━━━━━━━━━━━━━━━━━ 556s 1s/step - accuracy: 0.9098 - loss: 0.2402 - val_accuracy: 0.7992 - val_loss: 0.6249
Epoch 28/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 558s 1s/step - accuracy: 0.9176 - loss: 0.2224 - val_accuracy: 0.7956 - val_loss: 0.6526
Epoch 29/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 0s 1000ms/step - accuracy: 0.9217 - loss: 0.2105

546/546 ━━━━━━━━━━━━━━━━━━━━ 559s 1s/step - accuracy: 0.9217 - loss: 0.2105 - val_accuracy: 0.8010 - val_loss: 0.6828
Epoch 30/30
546/546 ━━━━━━━━━━━━━━━━━━━━ 553s 1s/step - accuracy: 0.9251 - loss: 0.2021 - val_accuracy: 0.8007 - val_loss: 0.7371
1213/1213 ━━━━━━━━━━━━━━━━━━━━ 77s 64ms/step
✅ Fold 3 Accuracy: 0.8156

✅ cnn_full finished. Mean Accuracy: 0.8124


-----------------------------------------